

### wrangle rota information
Transforms the raw normalised rota into a staffing-counts table:  
**one row per (date, shift, role) → headcount**

Key decisions:
- `lounge` is merged into `restaurant` (the lounge was always part of the same F&B operation and gets formally absorbed later).
- Only `working` records are counted — off, holiday, sick, etc. are excluded.
- Shift classification by `shift_end`: ends between 15:00–17:59 → **morning**, everything else → **afternoon**.

In [6]:
import pandas as pd



rota = pd.read_csv("../data/raw/rota_normalized.csv", parse_dates=["date"])
print(f"{len(rota):,} records  |  {rota['date'].min().date()} → {rota['date'].max().date()}")
rota.head(10)

24,576 records  |  2023-12-04 → 2026-04-26


,date,anonymous_id,role,service_area,shift_start,shift_end,status
0,2023-12-04,bartender_1,bartender,lounge,NaN,NaN,holiday
1,2023-12-04,bartender_2,bartender,lounge,NaN,NaN,off
2,2023-12-04,bartender_3,bartender,lounge,NaN,NaN,off
3,2023-12-04,bartender_4,bartender,lounge,16:00,00:30,working
4,2023-12-04,bartender_5,bartender,lounge,NaN,NaN,holiday
5,2023-12-04,bartender_6,bartender,lounge,16:00,00:30,working
6,2023-12-04,bartender_7,bartender,lounge,06:00,16:45,working
7,2023-12-04,bartender_8,bartender,lounge,09:00,15:30,working
8,2023-12-04,bartender_9,bartender,lounge,NaN,NaN,requested_off
9,2023-12-04,hostess_2,hostess,lounge,15:00,23:00,working


### 1. Filter to working records only

In [7]:
working = rota[rota["status"] == "working"].copy()
print(f"{len(working):,} working records  ({len(working)/len(rota)*100:.1f}% of total)")

19,360 working records  (78.8% of total)


### 2. Merge lounge → restaurant

This is especially important since the lounge was considered part of the restaurant in the beginning then later it branched out as a standalone service area and eventually got merged back into the restaurant.

For the sake of the thesis, we will assume that the lounge was always part of the restaurant from the beginning to the end.


In [8]:
print("Before merge:")
print(working["service_area"].value_counts())

working["service_area"] = working["service_area"].replace("lounge", "restaurant")

print("\nAfter merge:")
print(working["service_area"].value_counts())

Before merge:
service_area
restaurant    12373
bar            3681
lounge         3306
Name: count, dtype: int64

After merge:
service_area
restaurant    15679
bar            3681
Name: count, dtype: int64


## 3. Classify shifts — morning vs afternoon

Morning shifts end around 15:00–17:00 (the breakfast/lunch service). 

Some other shifts might be very short, ending between 9.00 and 14.00 as well, therefore considered morning shifts too. 

Everything else is an afternoon shift (dinner/evening/late-night).


In [9]:
# Drop the handful of working records with missing shift_end                                                                                                                                    
working = working[working["shift_end"].notna() & (working["shift_end"] != "")].copy()                                                                                                           
                                                                                                                                                                                                  
working["end_hour"] = pd.to_datetime(working["shift_end"], format="%H:%M").dt.hour                                                                                                              
                                                                                                                                                                                                  
print("shift_end hour distribution:")                                                                                                                                                           
print(working["end_hour"].value_counts().sort_index())   

shift_end hour distribution:
end_hour
0     3538
1      429
2      121
3       67
4      150
5       34
6        2
7       61
9        2
10       3
11      33
12     426
13     568
14     837
15    5261
16     835
17     368
18     162
19     143
20     434
21     216
22     450
23    5217
Name: count, dtype: int64


In [ ]:
working["shift"] = working["end_hour"].apply(
    lambda hour: "morning" if hour in (9, 10, 11, 12, 14, 15, 16, 17) else "afternoon"
)

print(working["shift"].value_counts())
print(f"\nMorning shift percentage: {(working['shift'] == 'morning').mean()*100:.1f}%")

shift
afternoon    11592
morning       7765
Name: count, dtype: int64

Morning pct: 40.1%


## 4. Aggregate — headcount per (date, shift, role)

This aggregate number helps us to know how many people, per role and eventually in total, worked per day.

In [11]:
staffing = (
    working
    .groupby(["date", "shift", "role"])
    .agg(headcount=("anonymous_id", "count"))
    .reset_index()
)

print(f"{len(staffing):,} rows  |  {staffing['date'].nunique()} unique dates")
staffing.head(20)

8,585 rows  |  854 unique dates


,date,shift,role,headcount
0,2023-12-04,afternoon,bartender,2
1,2023-12-04,afternoon,hostess,2
2,2023-12-04,afternoon,leader,1
3,2023-12-04,afternoon,runner,3
4,2023-12-04,afternoon,server,7
5,2023-12-04,morning,bartender,2
6,2023-12-04,morning,hostess,1
7,2023-12-04,morning,leader,1
8,2023-12-04,morning,runner,3
9,2023-12-04,morning,server,2


In [12]:
# Quick sanity check: average headcount by role and shift
pivot = staffing.pivot_table(
    index="role", columns="shift", values="headcount", aggfunc="mean"
).round(1)

print("Average daily headcount by role and shift:")
pivot

Average daily headcount by role and shift:


shift,afternoon,morning
role,,
bartender,3.6,1.3
hostess,1.4,1.2
leader,1.7,1.2
manager,1.5,1.1
runner,2.4,1.8
server,5.5,2.6


In [15]:
# Save
out_path = "../data_prep/outputs/rota_staffing_counts.csv"
staffing.to_csv(out_path, index=False)
print(f"Saved → {out_path}")
print(f"{len(staffing):,} rows  |  columns: {list(staffing.columns)}")

Saved → ../data_prep/outputs/rota_staffing_counts.csv
8,585 rows  |  columns: ['date', 'shift', 'role', 'headcount']


In [16]:
staffing[(staffing["role"] == "server") & (staffing["date"] == "2024-03-15")]

,date,shift,role,headcount
985,2024-03-15,afternoon,server,7
990,2024-03-15,morning,server,2
